# Demo 06 : A LangSmith experiment

Run the cells in order and watch.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.5", "langchain-core==1.6.10", "langchain-openai==1.7.1", "langgraph==1.2.15", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.24", "langchain-quickjs==0.3.9", "langsmith==0.14.8", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    THINKING_OFF = {"reasoning": {"effort": "none"}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# New OpenRouter accounts may send 20 requests a minute per model.
# One shared limiter keeps parallel agents under it ; raise COURSE_RPS when the cap lifts
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "0.3")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    kw.setdefault("temperature", 0)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · trainer

We create a small trainer dataset in the EU account and run an experiment with code evaluators. Account or permission failures are reported as failures; the local agent still runs. The dataset and experiment remain in the trainer account for review.

In [2]:
import json
import re
import sqlite3
from pathlib import Path

from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langchain.tools import tool

DB = Path(data("demo-world/service.db")).resolve()
MANUALS = Path(data_folder("demo-world/manuals"))


@tool
def part_stock(part_no: str) -> str:
    """Look up a Veenhaven Bus part's stock and depot by its part number."""
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        rows = db.execute("SELECT part_no, stock, depot FROM parts WHERE part_no = ?",
                          (part_no.upper(),)).fetchall()
    return json.dumps(rows)


@tool
def fault_guidance(error_code: str) -> str:
    """Read the meaning, first action and replacement condition for a bus error code."""
    return "\n".join(f"[{p.name}] {line}" for p in sorted(MANUALS.glob("*.md"))
                     for line in p.read_text(encoding="utf-8").splitlines()
                     if line.startswith("| " + error_code.upper() + " |")) or "No matching code."


QUESTION = "How many CE-CF-02 cooling fans are in stock and at which depot?"
SYSTEM = "Support Veenhaven Bus mechanics. Use the tools for facts. Preserve stock and depot names. Be brief."
LIMIT = {"recursion_limit": 20}


def ask(text):
    return {"messages": [{"role": "user", "content": text}]}


def show(result):
    for message in result["messages"]:
        if isinstance(message, dict):
            kind, content = message["type"], message.get("content", "")
            calls = message.get("tool_calls", [])
        else:
            kind, content, calls = message.type, message.content, getattr(message, "tool_calls", [])
        if kind in ("ai", "tool"):
            print(kind, ":", str(content)[:700], calls or "")

In [3]:
import os
from datetime import datetime, timezone

from langsmith import Client

agent = create_agent(llm, tools=[part_stock], system_prompt=SYSTEM)
local = agent.invoke(ask(QUESTION), LIMIT)
show(local)
status = {"endpoint": os.getenv("LANGSMITH_ENDPOINT"), "experiment": "not run"}
try:
    smith = Client(api_url=os.environ["LANGSMITH_ENDPOINT"], timeout_ms=20000)
    assert "eu.api.smith.langchain.com" in smith.api_url
    cases = [json.loads(line) for line in Path(data("demo-world/eval.jsonl")).read_text(encoding="utf-8").splitlines()]
    case = next(c for c in cases if c["id"] == "db-01")
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    dataset = smith.create_dataset(dataset_name="ari520-module09-" + stamp,
                                   description="Veenhaven Bus trainer stock smoke evaluation")
    smith.create_example(inputs={"question": case["question"]},
                         outputs={"part_no": "CE-DA-03", "stock": 0}, dataset_id=dataset.id)

    def predict(inputs):
        output = agent.invoke(ask(inputs["question"]), LIMIT)
        return {"answer": str(output["messages"][-1].content),
                "tools": [m.name for m in output["messages"] if m.type == "tool"]}

    def stock_score(outputs, reference_outputs):
        text = outputs["answer"].lower()
        return {"key": "stock_fact", "score": bool(re.search(r"\b(0|zero)\b", text)
                    or "out of stock" in text or "not in stock" in text)}

    def tool_score(outputs):
        return {"key": "tool_coverage", "score": "part_stock" in outputs["tools"]}

    experiment = smith.evaluate(predict, data=dataset.id, evaluators=[stock_score, tool_score],
                                experiment_prefix="ari520-module09", max_concurrency=1)
    rows = list(experiment)
    status["experiment"] = "completed"
    status["rows"] = len(rows)
    for row in rows:
        print("example :", row["example"].inputs["question"])
        print("answer :", row["run"].outputs["answer"])
        print("scores :", {score.key: score.score
                            for score in row["evaluation_results"]["results"]})
except Exception as exc:
    status["experiment"] = "FAILED"
    status["reason"] = type(exc).__name__
    print("LangSmith unavailable or experiment failed :", type(exc).__name__)
print("LangSmith status :", status)

ai :  [{'name': 'part_stock', 'args': {'part_no': 'CE-CF-02'}, 'id': 'call_9HUCOj1jYicqo0y443XXuNqJ', 'type': 'tool_call'}]
tool : [["CE-CF-02", 6, "Rietpoort"]] 
ai : There are 6 CE-CF-02 cooling fans in stock at Rietpoort. 


C:\Git\courseWare\Multi-Agents-LangGraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'ari520-module09-ab6e36ce' at:
https://eu.smith.langchain.com/o/273ccfef-9b92-471e-8d31-d52722357485/datasets/eea5a0c8-0ded-48ac-8380-ca6ebd530adf/compare?selectedSessions=07591bd6-26de-465f-9794-07cb061fb269




0it [00:00, ?it/s]

1it [00:38, 38.72s/it]

1it [00:38, 38.79s/it]

example : Is part CE-DA-03 in stock?
answer : No. CE-DA-03 is out of stock at Rietpoort.
scores : {'stock_fact': True, 'tool_coverage': True}
LangSmith status : {'endpoint': 'https://eu.api.smith.langchain.com', 'experiment': 'completed', 'rows': 1}
